In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import StandardScaler
df = pd.read_parquet("metropt3_preprocessed.parquet")
print(df.shape)

(1515288, 41)


In [ ]:
WARNING_HOURS = 6  

failures = [
    ("2020-04-18 00:00", "2020-04-18 23:59"),
    ("2020-05-29 23:30", "2020-05-30 06:00"),  
    ("2020-06-05 10:00", "2020-06-07 14:30"), 
    ("2020-07-15 14:30", "2020-07-15 19:00"),  
]

df["failure"] = 0
df["pre_failure"] = 0

for start, end in failures:
    start = pd.Timestamp(start)
    end = pd.Timestamp(end)
    warn_start = start - pd.Timedelta(hours=WARNING_HOURS)

    df.loc[df["timestamp"].between(start, end), "failure"] = 1

    pre_mask = (df["timestamp"] >= warn_start) & (df["timestamp"] < start)
    df.loc[pre_mask, "pre_failure"] = 1

    print(f"Failure {start}: pre_failure rows = {pre_mask.sum()}")

print(df[["failure", "pre_failure"]].sum())

Failure 2020-04-18 00:00:00: pre_failure rows = 1790
Failure 2020-05-29 23:30:00: pre_failure rows = 2179
Failure 2020-06-05 10:00:00: pre_failure rows = 2179
Failure 2020-07-15 14:30:00: pre_failure rows = 2179
failure        29934
pre_failure     8327
dtype: int64


In [15]:
split_date = pd.Timestamp("2020-06-01")

train = df[df["timestamp"] < split_date].copy()
test  = df[df["timestamp"] >= split_date].copy()

print("Train:", train.shape)
print("  failure rows:    ", train["failure"].sum())
print("  pre_failure rows:", train["pre_failure"].sum())
print("Test: ", test.shape)
print("  failure rows:    ", test["failure"].sum())
print("  pre_failure rows:", test["pre_failure"].sum())

Train: (855907, 43)
  failure rows:     11012
  pre_failure rows: 3969
Test:  (659381, 43)
  failure rows:     18922
  pre_failure rows: 4358


In [16]:
sensor_columns = ['TP2', 'TP3', 'H1', 'DV_pressure',
                  'Reservoirs', 'Oil_temperature', 'Motor_current']
rolling_columns = [c for c in df.columns if "_rolling_" in c]
rate_columns = [c for c in df.columns if c.endswith("_rate_of_change")]

continuous_columns = sensor_columns + rolling_columns + rate_columns
print(len(continuous_columns))

28


In [ ]:
from sklearn.preprocessing import StandardScaler

train_normal = train[(train["failure"] == 0) & (train["pre_failure"] == 0)]

scaler = StandardScaler()
scaler.fit(train_normal[continuous_columns])

train_scaled = train.copy()
test_scaled = test.copy()
train_scaled[continuous_columns] = scaler.transform(train[continuous_columns])
test_scaled[continuous_columns] = scaler.transform(test[continuous_columns])

print("Train normal rows used to fit:", len(train_normal))
print(train_scaled[continuous_columns].describe().T[["mean", "std"]].head(5))

Train normal rows used to fit: 840926
                 mean       std
TP2          0.030955  1.031572
TP3         -0.008070  0.995680
H1          -0.032601  1.033694
DV_pressure  0.071592  1.179663
Reservoirs  -0.008070  0.995683


In [18]:
import os
import joblib

os.makedirs("data/processed", exist_ok=True)
os.makedirs("models", exist_ok=True)

train_scaled.to_parquet("data/processed/train.parquet", index=False)
test_scaled.to_parquet("data/processed/test.parquet", index=False)
joblib.dump(scaler, "models/scaler.joblib")
joblib.dump(continuous_columns, "models/continuous_columns.joblib")

print("Saved.")

Saved.


In [19]:
for name, d in [("train", train_scaled), ("test", test_scaled)]:
    print(name, d.shape,
          "| NaN:", d.isna().sum().sum(),
          "| failure:", d["failure"].sum(),
          "| pre_failure:", d["pre_failure"].sum())

print("Train ends:", train_scaled["timestamp"].max())
print("Test starts:", test_scaled["timestamp"].min())

train (855907, 43) | NaN: 0 | failure: 11012 | pre_failure: 3969
test (659381, 43) | NaN: 0 | failure: 18922 | pre_failure: 4358
Train ends: 2020-05-31 23:49:44
Test starts: 2020-06-01 00:32:51
